# Challenge 2: Enhanced Gradient Boosting

Builds on the team's tuned `GradientBoostingClassifier` (Diego's notebook). Three changes, each checked against the original:

1. **Engineered features** (same four as the logistic regression notebook).
2. **Wider grid search.** The original grid's best values sat on its edges (`learning_rate` 0.03 and `n_estimators` 100 were the lowest tried, `max_depth` 4 the highest), so the true best may be outside it.
3. **A steadier threshold.** Instead of the single best-F1 cut-off, use the middle of the plateau where F1 is within 0.005 of the best.

Same split, folds and seed as every other team notebook. The 20% hold-out is **not** used here.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.base import clone
from sklearn.metrics import (make_scorer, f1_score, precision_score, recall_score,
                             roc_auc_score, average_precision_score)

RANDOM_STATE = 42
NAVY, ORANGE, TEAL, RED, GRAY = "#1f3b73", "#f28e2b", "#2a9d8f", "#d1495b", "#9aa0a6"
print("scikit-learn", sklearn.__version__)

## Load and split

Same code and seed as `challenge2_validation_split.ipynb`.

In [ ]:
train = pd.read_csv("challenge2_train.csv")

def to_01(col):
    """Turn True/False (bool or text) into 1/0."""
    if col.dtype == bool:
        return col.astype(int)
    return col.astype(str).str.strip().str.lower().map({"true": 1, "false": 0})

y = to_01(train["Revenue"])
X = train.drop(columns=["ID", "Revenue"]).copy()
X["Weekend"] = to_01(X["Weekend"])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y_true = y_train.to_numpy()
print(f"Training rows: {len(X_train):,} ({y_train.mean():.1%} buyers)")

## Features

`original` is exactly the team's gradient boosting input (72 columns). `engineered` adds four columns built from each row alone, so nothing leaks between folds: `HasPageValue`, `TotalPages`, `TotalDuration`, `AvgTimePerPage`. Trees do not need scaling or logs.

In [ ]:
raw_num_cols = ["Administrative", "Administrative_Duration", "Informational", "Informational_Duration",
                "ProductRelated", "ProductRelated_Duration", "PageValues",
                "BounceRates", "ExitRates", "SpecialDay"]
cat_cols = ["Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType"]
bin_cols = ["Weekend"]
new_cols = ["HasPageValue", "TotalPages", "TotalDuration", "AvgTimePerPage"]
assert set(raw_num_cols + cat_cols + bin_cols) == set(X.columns)

def add_features(df):
    d = df.copy()
    d["HasPageValue"] = (d["PageValues"] > 0).astype(int)
    d["TotalPages"] = d["Administrative"] + d["Informational"] + d["ProductRelated"]
    d["TotalDuration"] = (d["Administrative_Duration"] + d["Informational_Duration"]
                          + d["ProductRelated_Duration"])
    d["AvgTimePerPage"] = d["TotalDuration"] / d["TotalPages"].clip(lower=1)
    return d

def make_pipe(engineered, **gb_params):
    num = raw_num_cols + bin_cols + (new_cols if engineered else [])
    cols = ColumnTransformer([
        ("num", "passthrough", num),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ])
    steps = ([("features", FunctionTransformer(add_features))] if engineered else []) + [
        ("cols", cols),
        ("model", GradientBoostingClassifier(random_state=RANDOM_STATE, **gb_params)),
    ]
    return Pipeline(steps)

ORIGINAL = {"learning_rate": 0.03, "max_depth": 4, "n_estimators": 100, "subsample": 1.0}
print("Inputs, original:  ", make_pipe(False, **ORIGINAL)[:-1].fit_transform(X_train).shape[1])
print("Inputs, engineered:", make_pipe(True, **ORIGINAL)[:-1].fit_transform(X_train).shape[1])

## Step 1: do the engineered features help?

Same settings as the team's tuned model, with and without the four new columns. Scored on the 5 folds.

In [ ]:
scoring = {"f1": "f1", "roc_auc": "roc_auc", "pr_auc": "average_precision",
           "recall": "recall", "precision": make_scorer(precision_score, zero_division=0)}

def evaluate(name, pipe):
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, return_train_score=True, n_jobs=-1)
    row = {"model": name}
    for m in scoring:
        row[m] = res[f"test_{m}"].mean()
        row[f"{m} std"] = res[f"test_{m}"].std()
    row["train f1"] = res["train_f1"].mean()
    row["f1 gap"] = row["train f1"] - row["f1"]
    return row

SHOW = ["f1", "roc_auc", "pr_auc", "recall", "precision", "train f1", "f1 gap"]
results = [
    evaluate("GB, team settings", make_pipe(False, **ORIGINAL)),
    evaluate("GB, team settings + new features", make_pipe(True, **ORIGINAL)),
]
pd.DataFrame(results).set_index("model")[SHOW].round(3)

## Step 2: wider grid search

Search ranks by **ROC-AUC** (as in the other notebooks) and also records PR-AUC. `subsample` is fixed at 1.0, which won in the original grid. `min_samples_leaf` is new: it stops trees from making splits on a handful of visits. 144 combinations x 5 folds, a few minutes with `n_jobs=-1`.

In [ ]:
param_grid = {
    "model__learning_rate": [0.02, 0.03, 0.05],
    "model__n_estimators": [50, 100, 150, 200],
    "model__max_depth": [3, 4, 5, 6],
    "model__min_samples_leaf": [1, 10, 30],
}

search = GridSearchCV(make_pipe(True), param_grid, cv=cv,
                      scoring={"roc_auc": "roc_auc", "pr_auc": "average_precision"},
                      refit="roc_auc", n_jobs=-1, return_train_score=True)
search.fit(X_train, y_train)

best_params = {k.replace("model__", ""): v for k, v in search.best_params_.items()}
print("Best settings:", best_params)
print(f"Best CV ROC-AUC: {search.best_score_:.4f}")

In [ ]:
grid = pd.DataFrame(search.cv_results_)
for c in ["learning_rate", "n_estimators", "max_depth", "min_samples_leaf"]:
    grid[c] = grid[f"param_model__{c}"].astype(float)

top = (grid.sort_values("rank_test_roc_auc").head(10)
       [["learning_rate", "n_estimators", "max_depth", "min_samples_leaf",
         "mean_test_roc_auc", "std_test_roc_auc", "mean_test_pr_auc", "mean_train_roc_auc"]]
       .rename(columns={"mean_test_roc_auc": "CV ROC-AUC", "std_test_roc_auc": "CV std",
                        "mean_test_pr_auc": "CV PR-AUC", "mean_train_roc_auc": "train ROC-AUC"}))
top.round(4).reset_index(drop=True)

Check the edges again. If the best settings sit on the edge of this grid too, extend it once more. If they sit inside it, and the top rows are within one `CV std` of each other, the model is on a plateau and the exact choice matters little.

In [ ]:
edges = {"learning_rate": (0.02, 0.05), "n_estimators": (50, 200), "max_depth": (3, 6), "min_samples_leaf": (1, 30)}
for k, (lo, hi) in edges.items():
    v = best_params[k]
    print(f"{k:17s} best = {v:<6g} {'<- on the grid edge' if v in (lo, hi) else ''}")

In [ ]:
best_pipe = make_pipe(True, **best_params)
results.append(evaluate("GB, tuned + new features", best_pipe))
pd.DataFrame(results).set_index("model")[SHOW].round(3)

## Step 3: out-of-fold probabilities and threshold

The competition is scored on F1 for `Revenue=True`, so the cut-off matters. Every threshold from 0.05 to 0.95 is tried on the out-of-fold predictions. Two choices are reported: the single best-F1 threshold and the **plateau centre** (the middle of all thresholds within 0.005 F1 of the best), which is less likely to be a lucky spike.

In [ ]:
oof = np.zeros(len(y_train))
for tr, va in cv.split(X_train, y_train):
    m = clone(best_pipe).fit(X_train.iloc[tr], y_train.iloc[tr])
    oof[va] = m.predict_proba(X_train.iloc[va])[:, 1]

print(f"Out-of-fold ROC-AUC: {roc_auc_score(y_true, oof):.3f}")
print(f"Out-of-fold PR-AUC:  {average_precision_score(y_true, oof):.3f}")

thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame({
    "threshold": thresholds,
    "f1": [f1_score(y_true, oof >= t) for t in thresholds],
    "precision": [precision_score(y_true, oof >= t, zero_division=0) for t in thresholds],
    "recall": [recall_score(y_true, oof >= t) for t in thresholds],
})
best_f1 = curve["f1"].max()
thr_best = float(curve.loc[curve["f1"].idxmax(), "threshold"])
plateau = curve[curve["f1"] >= best_f1 - 0.005]["threshold"]
thr_plateau = float(round(plateau.median(), 2))

def at(t):
    p = oof >= t
    return {"threshold": t, "f1": f1_score(y_true, p), "precision": precision_score(y_true, p, zero_division=0),
            "recall": recall_score(y_true, p)}

print(f"Plateau: thresholds {plateau.min():.2f} to {plateau.max():.2f}")
pd.DataFrame([at(0.5), at(thr_best), at(thr_plateau)], index=["default 0.5", "best F1", "plateau centre"]).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(curve["threshold"], curve["f1"], color=NAVY, linewidth=2.5, label="F1")
ax.plot(curve["threshold"], curve["precision"], color=TEAL, linewidth=1.5, label="precision")
ax.plot(curve["threshold"], curve["recall"], color=ORANGE, linewidth=1.5, label="recall")
ax.axvspan(plateau.min(), plateau.max(), color=NAVY, alpha=0.08, label="within 0.005 of best F1")
ax.axvline(thr_plateau, color=RED, linestyle="--", label=f"chosen {thr_plateau:.2f}")
ax.set_xlabel("threshold"); ax.set_ylabel("out-of-fold score"); ax.set_title("F1 by threshold, enhanced gradient boosting")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.17))
plt.tight_layout(); plt.show()

## Summary and settings to carry into the final retrain

In [ ]:
orig_oof = np.zeros(len(y_train))
for tr, va in cv.split(X_train, y_train):
    m = clone(make_pipe(False, **ORIGINAL)).fit(X_train.iloc[tr], y_train.iloc[tr])
    orig_oof[va] = m.predict_proba(X_train.iloc[va])[:, 1]
o_curve = [f1_score(y_true, orig_oof >= t) for t in thresholds]
o_thr = float(thresholds[int(np.argmax(o_curve))])

s = at(thr_plateau)
compare = pd.DataFrame([
    {"model": "GB, team version", "threshold": o_thr, "f1": max(o_curve),
     "roc_auc": roc_auc_score(y_true, orig_oof), "pr_auc": average_precision_score(y_true, orig_oof)},
    {"model": "GB, enhanced", "threshold": thr_plateau, "f1": s["f1"],
     "roc_auc": roc_auc_score(y_true, oof), "pr_auc": average_precision_score(y_true, oof)},
]).set_index("model")
display(compare.round(3))

final_settings = {**best_params, "threshold": thr_plateau,
                  "features": "original + HasPageValue, TotalPages, TotalDuration, AvgTimePerPage"}
final_settings

## Reading the result

- Compare the two rows above. The enhanced model only counts as better if it beats the team version by more than the fold-to-fold spread in Step 1 (`f1 std`, `roc_auc std`). A gain of 0.005 F1 is noise.
- The enhanced model was picked from a bigger grid, so its out-of-fold score is a bit more optimistic than the team version's. The hold-out score, run once after the team compares all models, is the fair test.
- Carry `final_settings` (settings, features and threshold) into the final retrain on all training rows.